# 🖼️ Classificação de imagens com CIFAR-100

Treina uma rede neural convolucional capaz de classificar imagens em **100
categorias diferentes** — de maçã a zebra, de foguete a girassol.

**Framework:** Keras / TensorFlow
**Dataset:** CIFAR-100 (embutido no Keras, baixa sozinho)
**Ambiente:** Google Colab com GPU

---

## ⚙️ Antes de começar

**Ative a GPU:** `Ambiente de execução → Alterar o tipo de ambiente de execução
→ Acelerador de hardware: GPU (T4) → Salvar`

Depois é só rodar as células em ordem, de cima para baixo.

**Tempo estimado:** 12 a 18 minutos na GPU T4 gratuita.

---

## O diferencial deste projeto

Quase todo tutorial de CIFAR-100 termina em "acurácia de X%". Aqui vamos além,
aproveitando uma característica pouco explorada do dataset: além das 100
classes, ele traz **20 superclasses** que agrupam as categorias por afinidade.

> `dolphin`, `whale`, `otter`, `seal` e `beaver` pertencem todos à superclasse
> `aquatic_mammals`.

Isso permite responder uma pergunta muito mais interessante que a acurácia:
**quando o modelo erra, ele erra feio ou erra perto?** Confundir golfinho com
baleia é um erro bem diferente de confundir golfinho com guarda-roupa.

---

## Roteiro

| Etapa | O que acontece |
|---|---|
| 1 | Prepara o ambiente e confirma a GPU |
| 2 | Carrega e explora o CIFAR-100 |
| 3 | Monta a arquitetura da rede |
| 4 | Treina o modelo |
| 5 | Avalia com acurácia top-1 e top-5 |
| 6 | **Analisa os erros usando as superclasses** |
| 7 | Testa em imagens individuais |
| 8 | Salva e baixa o modelo |

---
# 1. Preparação

O TensorFlow já vem instalado no Colab, então não é preciso instalar nada. Só
confirmamos a versão e a presença da GPU.

Fixamos também as sementes aleatórias. Sem isso, duas execuções do mesmo código
dão resultados diferentes, e fica impossível saber se uma mudança melhorou o
modelo ou se foi só sorte.

In [ ]:
import random

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEMENTE = 42
random.seed(SEMENTE)
np.random.seed(SEMENTE)
tf.random.set_seed(SEMENTE)

print(f"TensorFlow : {tf.__version__}")
print(f"Keras      : {keras.__version__}")

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"GPU        : {gpus[0].name}")
    print("\nTudo certo, pode seguir.")
else:
    print("GPU        : nenhuma detectada")
    print("\n[!] Sem GPU o treino leva mais de uma hora.")
    print("    Ambiente de execução > Alterar o tipo de ambiente de execução")
    print("    > Acelerador de hardware: GPU (T4) > Salvar")

---
# 2. O dataset CIFAR-100

| | |
|---|---|
| **Imagens** | 60.000 coloridas de 32×32 pixels |
| **Treino** | 50.000 imagens (500 por classe) |
| **Teste** | 10.000 imagens (100 por classe) |
| **Classes finas** | 100 |
| **Superclasses** | 20 (5 classes finas em cada) |

O dataset é **perfeitamente balanceado**: exatamente 500 imagens de treino por
classe. Isso elimina uma das fontes mais comuns de viés, mas cria outra
dificuldade: 500 exemplos por categoria é pouco para distinguir 100 classes.

## Carregando com dois níveis de rótulo

O Keras oferece o CIFAR-100 em dois modos, `fine` (100 classes) e `coarse`
(20 superclasses). Carregamos os **dois**: as imagens são as mesmas, na mesma
ordem, então cada imagem fica com seus dois rótulos alinhados.

É assim que descobrimos qual superclasse corresponde a cada classe fina — sem
precisar escrever uma tabela de 100 linhas à mão, que seria uma fonte garantida
de erro de digitação.

In [ ]:
from tensorflow.keras.datasets import cifar100

# Mesmas imagens, dois níveis de rótulo
(x_treino, y_fino_treino), (x_teste, y_fino_teste) = cifar100.load_data(
    label_mode="fine")
(_, y_grosso_treino), (_, y_grosso_teste) = cifar100.load_data(
    label_mode="coarse")

# Achatamos de (N, 1) para (N,), que é o formato esperado adiante
y_fino_treino = y_fino_treino.ravel()
y_fino_teste = y_fino_teste.ravel()
y_grosso_treino = y_grosso_treino.ravel()
y_grosso_teste = y_grosso_teste.ravel()

print(f"Treino : {x_treino.shape}  rótulos {y_fino_treino.shape}")
print(f"Teste  : {x_teste.shape}  rótulos {y_fino_teste.shape}")
print(f"Pixels : {x_treino.dtype}, de {x_treino.min()} a {x_treino.max()}")
print(f"\nClasses finas .. {len(np.unique(y_fino_treino))}")
print(f"Superclasses ... {len(np.unique(y_grosso_treino))}")

contagem = np.bincount(y_fino_treino)
print(f"\nImagens por classe: mínimo {contagem.min()}, máximo {contagem.max()}")
print("Dataset perfeitamente balanceado." if contagem.min() == contagem.max()
      else "[!] Dataset desbalanceado.")

### Os nomes das classes

O Keras entrega apenas números de 0 a 99. Os nomes vêm da especificação
original do CIFAR-100, em **ordem alfabética** — que é exatamente a ordem dos
índices.

In [ ]:
NOMES_FINOS = [
    "apple", "aquarium_fish", "baby", "bear", "beaver", "bed", "bee", "beetle",
    "bicycle", "bottle", "bowl", "boy", "bridge", "bus", "butterfly", "camel",
    "can", "castle", "caterpillar", "cattle", "chair", "chimpanzee", "clock",
    "cloud", "cockroach", "couch", "crab", "crocodile", "cup", "dinosaur",
    "dolphin", "elephant", "flatfish", "forest", "fox", "girl", "hamster",
    "house", "kangaroo", "keyboard", "lamp", "lawn_mower", "leopard", "lion",
    "lizard", "lobster", "man", "maple_tree", "motorcycle", "mountain", "mouse",
    "mushroom", "oak_tree", "orange", "orchid", "otter", "palm_tree", "pear",
    "pickup_truck", "pine_tree", "plain", "plate", "poppy", "porcupine",
    "possum", "rabbit", "raccoon", "ray", "road", "rocket", "rose", "sea",
    "seal", "shark", "shrew", "skunk", "skyscraper", "snail", "snake", "spider",
    "squirrel", "streetcar", "sunflower", "sweet_pepper", "table", "tank",
    "telephone", "television", "tiger", "tractor", "train", "trout", "tulip",
    "turtle", "wardrobe", "whale", "willow_tree", "wolf", "woman", "worm",
]

NOMES_GROSSOS = [
    "aquatic_mammals", "fish", "flowers", "food_containers",
    "fruit_and_vegetables", "household_electrical_devices",
    "household_furniture", "insects", "large_carnivores",
    "large_man-made_outdoor_things", "large_natural_outdoor_scenes",
    "large_omnivores_and_herbivores", "medium_mammals",
    "non-insect_invertebrates", "people", "reptiles", "small_mammals",
    "trees", "vehicles_1", "vehicles_2",
]

assert len(NOMES_FINOS) == 100 and len(NOMES_GROSSOS) == 20
print(f"{len(NOMES_FINOS)} classes finas, {len(NOMES_GROSSOS)} superclasses")
print("\nPrimeiras 10 classes:", ", ".join(NOMES_FINOS[:10]))

### Descobrindo a hierarquia a partir dos dados

Como cada imagem tem os dois rótulos, basta percorrer o conjunto de treino uma
vez para montar o mapa `classe fina → superclasse`. Nenhuma tabela escrita à
mão, nenhum risco de erro de digitação.

In [ ]:
from collections import defaultdict

# Mapa construído a partir dos próprios dados
FINO_PARA_GROSSO = {}
for f, g in zip(y_fino_treino, y_grosso_treino):
    FINO_PARA_GROSSO[int(f)] = int(g)

assert len(FINO_PARA_GROSSO) == 100, "nem toda classe fina foi mapeada"

# Agrupamento inverso, para exibição
GROSSO_PARA_FINOS = defaultdict(list)
for f, g in sorted(FINO_PARA_GROSSO.items()):
    GROSSO_PARA_FINOS[g].append(NOMES_FINOS[f])

print("AS 20 SUPERCLASSES E SUAS 5 CLASSES\n")
for g in range(20):
    print(f"{NOMES_GROSSOS[g]:<32} {', '.join(GROSSO_PARA_FINOS[g])}")

Repare como algumas superclasses reúnem categorias **visualmente parecidas**
(`aquatic_mammals`, `trees`) e outras reúnem coisas que só têm em comum a
função (`household_electrical_devices` junta relógio, teclado, lâmpada,
telefone e televisão). Isso vai importar na análise de erros.

### Olhando as imagens

32×32 pixels é **muito pouco**. Vale ver com os próprios olhos antes de julgar
o desempenho do modelo: várias dessas imagens são difíceis até para uma pessoa.

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(SEMENTE)
indices = rng.choice(len(x_treino), size=24, replace=False)

fig, axes = plt.subplots(3, 8, figsize=(16, 6.5))
for ax, i in zip(axes.ravel(), indices):
    ax.imshow(x_treino[i])
    ax.axis("off")
    fino = NOMES_FINOS[y_fino_treino[i]]
    grosso = NOMES_GROSSOS[y_grosso_treino[i]]
    ax.set_title(f"{fino}\n({grosso})", fontsize=7.5)

fig.suptitle("Amostras do CIFAR-100 — classe fina e superclasse",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
# 3. A arquitetura da rede

## Por que treinar do zero, e não usar um modelo pré-treinado

Modelos pré-treinados no ImageNet (ResNet, EfficientNet) esperam imagens de
224×224. Nossas imagens têm 32×32 — ampliá-las até esse tamanho gera um borrão
e desperdiça a maior parte do cálculo em pixels interpolados, inventados.

Para imagens pequenas, uma CNN treinada do zero é o caminho padrão: rápida,
leve e adequada à resolução. (Transfer learning **também funciona** no
CIFAR-100 e chega mais longe em acurácia, ao custo de bem mais tempo — está
listado nas extensões, ao final.)

## A estrutura escolhida

Três blocos convolucionais, cada um com o mesmo padrão:

```
Conv2D → BatchNorm → ReLU        (duas vezes)
MaxPooling2D                      reduz a imagem pela metade
Dropout                           desliga neurônios ao acaso
```

| Bloco | Filtros | Resolução | Dropout |
|---|---|---|---|
| 1 | 64 | 32×32 → 16×16 | 20% |
| 2 | 128 | 16×16 → 8×8 | 30% |
| 3 | 256 | 8×8 → 4×4 | 40% |

O número de filtros **dobra** a cada bloco enquanto a resolução cai pela
metade. É o padrão clássico: nas primeiras camadas a rede aprende poucos
padrões simples (bordas, cores); nas últimas, muitos padrões complexos
(texturas, partes de objeto).

## As três decisões que mais importam

**BatchNormalization** normaliza as ativações a cada lote. Sem ela, redes
profundas treinam devagar e travam com facilidade. Note que usamos
`use_bias=False` nas convoluções: o BatchNorm já tem seu próprio termo de
deslocamento, e o viés da convolução seria redundante.

**Dropout crescente** desliga neurônios ao acaso durante o treino, forçando a
rede a não depender de nenhum caminho específico. Ele aumenta de 20% para 40%
ao longo da rede porque as camadas finais, com mais parâmetros, decoram mais.

**GlobalAveragePooling** em vez de `Flatten`. O `Flatten` de um mapa 4×4×256
geraria 4.096 valores e uma camada densa com milhões de parâmetros. O pooling
global tira a média de cada mapa de ativação e entrega 256 valores — muito
menos parâmetros e menos overfitting.

## Data augmentation dentro do modelo

As camadas de augmentation ficam **dentro** do modelo, e não no pipeline de
dados. Duas vantagens: elas se desligam sozinhas na avaliação e na predição
(o Keras cuida disso), e vão junto quando o modelo é salvo.

- `RandomFlip("horizontal")` — espelha; um cachorro espelhado continua cachorro
- `RandomTranslation(0.1)` — desloca até 10%; o objeto nem sempre está centrado
- `RandomZoom(0.1)` — aproxima ou afasta até 10%

Não usamos flip **vertical**: um carro de cabeça para baixo não é uma imagem
realista, e treinar com isso atrapalha em vez de ajudar.

In [ ]:
def bloco_convolucional(x, filtros, dropout):
    """Dois Conv2D + BatchNorm, seguidos de pooling e dropout."""
    for _ in range(2):
        # use_bias=False porque o BatchNorm logo abaixo já tem deslocamento
        x = layers.Conv2D(filtros, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(dropout)(x)
    return x


def construir_modelo(n_classes=100):
    entrada = keras.Input(shape=(32, 32, 3), name="imagem")

    # Augmentation: ativa só no treino, desliga sozinha na avaliação
    x = layers.RandomFlip("horizontal")(entrada)
    x = layers.RandomTranslation(0.1, 0.1)(x)
    x = layers.RandomZoom(0.1)(x)

    # Pixels de 0-255 para 0-1
    x = layers.Rescaling(1.0 / 255)(x)

    x = bloco_convolucional(x, 64, 0.20)
    x = bloco_convolucional(x, 128, 0.30)
    x = bloco_convolucional(x, 256, 0.40)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.5)(x)
    saida = layers.Dense(n_classes, activation="softmax", name="classe")(x)

    return keras.Model(entrada, saida, name="cnn_cifar100")


modelo = construir_modelo()
modelo.summary()
print(f"\nTotal de parâmetros: {modelo.count_params():,}")

---
# 4. Treino

## A função de perda

Usamos `sparse_categorical_crossentropy`. O prefixo **sparse** significa que os
rótulos são inteiros (`37`) em vez de vetores one-hot (`[0,...,1,...,0]`). Como
o Keras entrega o CIFAR-100 já em inteiros, essa é a escolha natural — e
economiza memória, já que um vetor de 100 posições por imagem seria puro
desperdício.

## As duas métricas

| Métrica | O que mede |
|---|---|
| **Acurácia (top-1)** | O palpite mais provável estava certo? |
| **Acurácia top-5** | A resposta certa estava entre os 5 palpites mais prováveis? |

Top-5 é relevante aqui porque com 100 classes e imagens de 32×32 há ambiguidade
real. Um modelo que coloca a resposta certa em segundo lugar é muito mais útil
que um que a coloca em octogésimo — e a acurácia top-1 trata os dois como erro
igual. Para referência, o acerto aleatório é **1% em top-1 e 5% em top-5**.

## Os três callbacks

| Callback | Função |
|---|---|
| `EarlyStopping` | Para quando a acurácia de validação estaciona por 12 épocas, e **restaura os melhores pesos** |
| `ReduceLROnPlateau` | Corta a taxa de aprendizado pela metade quando a perda estagna, permitindo ajuste fino |
| `ModelCheckpoint` | Salva o melhor modelo em disco, não o último |

A distinção entre *melhor* e *último* importa: a última época frequentemente é
pior que alguma anterior, porque o modelo já começou a decorar.

In [ ]:
modelo = construir_modelo()

modelo.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5"),
    ],
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=12,
        restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=5,
        min_lr=1e-6, verbose=1),
    keras.callbacks.ModelCheckpoint(
        "melhor_modelo.keras", monitor="val_accuracy",
        save_best_only=True, verbose=0),
]

print("Iniciando o treino. Cada época leva cerca de 10 a 15 segundos na T4.\n")

historico = modelo.fit(
    x_treino, y_fino_treino,
    validation_data=(x_teste, y_fino_teste),
    epochs=60,
    batch_size=128,
    callbacks=callbacks,
    verbose=2,      # uma linha por época, sem barra de progresso
)

print(f"\nTreino concluído em {len(historico.history['loss'])} épocas.")

### As curvas de treino

O que procurar:

- **As duas curvas de perda caindo juntas** é o cenário saudável.
- **Perda de treino caindo enquanto a de validação sobe** é overfitting: o
  modelo está decorando as imagens de treino.
- **Degraus na curva** coincidem com as reduções da taxa de aprendizado feitas
  pelo `ReduceLROnPlateau` — é normal e geralmente vem acompanhado de uma
  melhora.

Neste modelo a perda de validação costuma ficar **abaixo** da de treino nas
primeiras épocas. Não é erro: o dropout e a augmentation só atuam no treino, o
que o torna artificialmente mais difícil que a validação.

In [ ]:
import matplotlib.pyplot as plt

h = historico.history
epocas = range(1, len(h["loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))

axes[0].plot(epocas, h["loss"], label="treino", color="#2E86AB")
axes[0].plot(epocas, h["val_loss"], label="validação", color="#C0392B")
axes[0].set_title("Perda", fontweight="bold")
axes[0].set_xlabel("época")
axes[0].legend()

axes[1].plot(epocas, h["accuracy"], label="treino", color="#2E86AB")
axes[1].plot(epocas, h["val_accuracy"], label="validação", color="#C0392B")
axes[1].axhline(0.01, color="gray", linestyle=":", label="acaso (1%)")
axes[1].set_title("Acurácia top-1", fontweight="bold")
axes[1].set_xlabel("época")
axes[1].legend()

axes[2].plot(epocas, h["top5"], label="treino", color="#2E86AB")
axes[2].plot(epocas, h["val_top5"], label="validação", color="#C0392B")
axes[2].axhline(0.05, color="gray", linestyle=":", label="acaso (5%)")
axes[2].set_title("Acurácia top-5", fontweight="bold")
axes[2].set_xlabel("época")
axes[2].legend()

for ax in axes:
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

melhor = int(np.argmax(h["val_accuracy"])) + 1
print(f"Melhor época: {melhor} "
      f"(acurácia de validação {max(h['val_accuracy']):.4f})")

---
# 5. Avaliação

Medindo o desempenho no conjunto de teste — 10.000 imagens que o modelo nunca
viu durante o treino.

In [ ]:
resultado = modelo.evaluate(x_teste, y_fino_teste, verbose=0, return_dict=True)

print("=" * 52)
print("DESEMPENHO NO CONJUNTO DE TESTE")
print("=" * 52)
print(f"  Perda ............ {resultado['loss']:.4f}")
print(f"  Acurácia top-1 ... {resultado['accuracy']*100:.2f}%")
print(f"  Acurácia top-5 ... {resultado['top5']*100:.2f}%")
print("=" * 52)
print(f"\nPara referência, o acerto aleatório seria 1,00% (top-1) e 5,00% (top-5).")
print(f"O modelo é {resultado['accuracy']/0.01:.0f}x melhor que o acaso.")

In [ ]:
# Predições para todas as análises seguintes
probabilidades = modelo.predict(x_teste, batch_size=256, verbose=0)
y_pred = probabilidades.argmax(axis=1)

print(f"Predições: {probabilidades.shape}")
print(f"Acertos:   {(y_pred == y_fino_teste).sum()} de {len(y_fino_teste)}")

### As classes mais fáceis e mais difíceis

Com 100 classes, a média esconde muita coisa. Algumas categorias o modelo
acerta quase sempre; outras são quase um chute.

In [ ]:
# O CIFAR-100 garante 100 imagens de teste por classe. Deixamos a premissa
# explícita: sem ela, uma classe vazia produziria 'nan' silenciosamente.
n_por_classe = np.bincount(y_fino_teste, minlength=100)
assert n_por_classe.min() > 0, "há classe fina sem imagens de teste"

acuracia_por_classe = np.array([
    (y_pred[y_fino_teste == c] == c).mean() for c in range(100)
])

ordem = np.argsort(acuracia_por_classe)

print("10 CLASSES MAIS FÁCEIS")
print("-" * 40)
for c in ordem[::-1][:10]:
    print(f"  {NOMES_FINOS[c]:<18} {acuracia_por_classe[c]*100:5.1f}%")

print("\n10 CLASSES MAIS DIFÍCEIS")
print("-" * 40)
for c in ordem[:10]:
    print(f"  {NOMES_FINOS[c]:<18} {acuracia_por_classe[c]*100:5.1f}%")

print(f"\nMédia {acuracia_por_classe.mean()*100:.1f}% · "
      f"melhor {acuracia_por_classe.max()*100:.1f}% · "
      f"pior {acuracia_por_classe.min()*100:.1f}%")

### Os pares mais confundidos

Uma matriz de confusão 100×100 é ilegível. Em vez do gráfico, listamos
diretamente os pares que o modelo mais troca — muito mais informativo.

In [ ]:
from collections import Counter

confusoes = Counter()
for verdadeiro, previsto in zip(y_fino_teste, y_pred):
    if verdadeiro != previsto:
        confusoes[(int(verdadeiro), int(previsto))] += 1

print("OS 15 ERROS MAIS FREQUENTES")
print("-" * 62)
print(f"  {'classe real':<18}{'previsto como':<18}{'vezes':>7}  {'mesma super?':>12}")
print("-" * 62)

for (real, previsto), n in confusoes.most_common(15):
    mesma = FINO_PARA_GROSSO[real] == FINO_PARA_GROSSO[previsto]
    marca = "sim" if mesma else "não"
    print(f"  {NOMES_FINOS[real]:<18}{NOMES_FINOS[previsto]:<18}{n:>7}  {marca:>12}")

---
# 6. 🔍 A análise que importa: erros por superclasse

Chegamos ao ponto central do projeto.

Suponha que o modelo acerte 60% das imagens. Os 40% restantes são todos iguais?
Não. Existe uma diferença enorme entre estes dois erros:

| Erro | Gravidade |
|---|---|
| `dolphin` classificado como `whale` | Pequeno — ambos são `aquatic_mammals` |
| `dolphin` classificado como `wardrobe` | Grave — o modelo não entendeu nada |

A acurácia trata os dois como o mesmo erro. A hierarquia do CIFAR-100 permite
separá-los, e responder: **quando erra, o modelo pelo menos chega perto?**

Medimos três coisas:

1. **Acurácia na classe fina** — o que já calculamos
2. **Acurácia na superclasse** — colapsando as predições em 20 grupos
3. **Taxa de erro "próximo"** — entre os erros, quantos ficaram na superclasse certa

In [ ]:
# Traduz classes finas em superclasses
para_super = np.vectorize(lambda c: FINO_PARA_GROSSO[int(c)])

super_real = para_super(y_fino_teste)
super_prev = para_super(y_pred)

acuracia_fina = (y_pred == y_fino_teste).mean()
acuracia_super = (super_prev == super_real).mean()

erros = y_fino_teste != y_pred
n_erros = int(erros.sum())
erros_proximos = int((super_real[erros] == super_prev[erros]).sum())

print("=" * 58)
print("ANÁLISE HIERÁRQUICA DOS ERROS")
print("=" * 58)
print(f"  Acurácia na classe fina (100) ...... {acuracia_fina*100:5.2f}%")
print(f"  Acurácia na superclasse (20) ....... {acuracia_super*100:5.2f}%")
print(f"  Ganho ao relaxar a exigência ....... {(acuracia_super-acuracia_fina)*100:5.2f} pontos")
print()
print(f"  Total de erros ..................... {n_erros}")
print(f"  Erros na superclasse CERTA ......... {erros_proximos:5d} "
      f"({erros_proximos/n_erros*100:.1f}%)")
print(f"  Erros na superclasse ERRADA ........ {n_erros-erros_proximos:5d} "
      f"({(n_erros-erros_proximos)/n_erros*100:.1f}%)")
print("=" * 58)

esperado_ao_acaso = 4 / 99 * 100    # 4 outras classes da mesma super, de 99
print(f"\nSe os erros fossem aleatórios, apenas {esperado_ao_acaso:.1f}% cairiam")
print("na superclasse certa (4 classes irmãs entre as 99 alternativas).")

razao = (erros_proximos / n_erros * 100) / esperado_ao_acaso
print(f"O modelo erra {razao:.1f}x mais 'perto' do que o acaso explicaria.")

> **Como ler esse número.** Se os erros fossem aleatórios, só cerca de **4%**
> cairiam na superclasse correta — há 4 classes irmãs entre 99 alternativas
> possíveis. Qualquer valor bem acima disso prova que o modelo **aprendeu
> estrutura semântica**, e não apenas memorizou aparências: quando não sabe
> exatamente o que é, ele ainda acerta a família.
>
> Isso é algo que a acurácia sozinha nunca mostraria.

### Desempenho por superclasse

Quais famílias o modelo domina e quais o confundem?

In [ ]:
n_por_super = np.bincount(super_real, minlength=20)
assert n_por_super.min() > 0, "há superclasse sem imagens de teste"

acuracia_por_super = np.array([
    (super_prev[super_real == g] == g).mean() for g in range(20)
])

ordem = np.argsort(acuracia_por_super)

plt.figure(figsize=(11, 7))
cores = plt.cm.RdYlGn(acuracia_por_super[ordem])
barras = plt.barh([NOMES_GROSSOS[g] for g in ordem],
                  acuracia_por_super[ordem] * 100,
                  color=cores, edgecolor="white")
plt.bar_label(barras, fmt="%.1f%%", fontsize=9, padding=3)
plt.axvline(acuracia_super * 100, color="#C0392B", linestyle="--", linewidth=1.6)
plt.title("Acurácia por superclasse\n"
          f"(linha vermelha = média geral, {acuracia_super*100:.1f}%)",
          fontweight="bold")
plt.xlabel("acurácia (%)")
plt.xlim(0, 108)
plt.tight_layout()
plt.show()

print(f"Superclasse mais fácil:  {NOMES_GROSSOS[ordem[-1]]} "
      f"({acuracia_por_super[ordem[-1]]*100:.1f}%)")
print(f"Superclasse mais difícil: {NOMES_GROSSOS[ordem[0]]} "
      f"({acuracia_por_super[ordem[0]]*100:.1f}%)")

Vale comparar este gráfico com a lista de superclasses da etapa 2. As famílias
**visualmente coerentes** (`trees`, `flowers`, `people`) tendem a ir bem, porque
seus membros compartilham forma e cor. Já as famílias definidas por **função**
(`household_electrical_devices`, que junta relógio, teclado, lâmpada, telefone
e televisão) costumam ir pior: nada nesses objetos se parece visualmente, e uma
rede convolucional só enxerga aparência.

---
# 7. Testando em imagens individuais

Além de acertar ou errar, vale olhar a **confiança** do modelo. Um erro com 95%
de confiança é bem mais preocupante que um erro com 30% — no segundo caso, o
modelo pelo menos sinalizou que estava em dúvida.

In [ ]:
rng = np.random.default_rng(7)
amostra = rng.choice(len(x_teste), size=12, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(14, 11))
for ax, i in zip(axes.ravel(), amostra):
    ax.imshow(x_teste[i])
    ax.axis("off")

    real = int(y_fino_teste[i])
    previsto = int(y_pred[i])
    confianca = float(probabilidades[i, previsto])
    acertou = real == previsto
    mesma_super = FINO_PARA_GROSSO[real] == FINO_PARA_GROSSO[previsto]

    if acertou:
        cor, marca = "#1E8449", "correto"
    elif mesma_super:
        cor, marca = "#F18F01", "erro próximo"
    else:
        cor, marca = "#C0392B", "erro distante"

    ax.set_title(
        f"real: {NOMES_FINOS[real]}\n"
        f"previu: {NOMES_FINOS[previsto]} ({confianca*100:.0f}%)\n"
        f"{marca}",
        fontsize=9, color=cor, fontweight="bold")

fig.suptitle("Verde = acertou · Laranja = errou mas ficou na mesma superclasse · "
             "Vermelho = errou longe", fontsize=11, y=1.00)
plt.tight_layout()
plt.show()

### Os 5 palpites do modelo

Para uma imagem específica, vale ver a distribuição inteira de probabilidades.
Ela revela *com o que* o modelo hesitou.

In [ ]:
i = int(amostra[0])

top5 = np.argsort(probabilidades[i])[::-1][:5]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5),
                         gridspec_kw={"width_ratios": [1, 2]})

axes[0].imshow(x_teste[i])
axes[0].axis("off")
axes[0].set_title(f"Real: {NOMES_FINOS[y_fino_teste[i]]}", fontweight="bold")

nomes = [NOMES_FINOS[c] for c in top5][::-1]
valores = [probabilidades[i, c] * 100 for c in top5][::-1]
cores = ["#1E8449" if c == y_fino_teste[i] else "#7FB3D5" for c in top5][::-1]

barras = axes[1].barh(nomes, valores, color=cores, edgecolor="white")
axes[1].bar_label(barras, fmt="%.1f%%", fontsize=10, padding=3)
axes[1].set_title("Os 5 palpites mais prováveis", fontweight="bold")
axes[1].set_xlabel("probabilidade (%)")
axes[1].set_xlim(0, max(valores) * 1.22)

plt.tight_layout()
plt.show()

---
# 8. Salvando o modelo

O formato `.keras` guarda arquitetura, pesos e configuração de treino num único
arquivo. Como as camadas de augmentation fazem parte do modelo, elas vão junto
— quem carregar o arquivo recebe exatamente o mesmo comportamento.

In [ ]:
from pathlib import Path

ARQUIVO = "modelo_cifar100.keras"
modelo.save(ARQUIVO)

tamanho = Path(ARQUIVO).stat().st_size / 1e6
print(f"Modelo salvo: {ARQUIVO} ({tamanho:.1f} MB)")

# Conferindo que o modelo recarregado se comporta igual
recarregado = keras.models.load_model(ARQUIVO)
teste = recarregado.predict(x_teste[:100], verbose=0).argmax(axis=1)
identico = (teste == y_pred[:100]).all()
print("Modelo recarregado produz as mesmas predições:", identico)

In [ ]:
from pathlib import Path

from google.colab import files

for arquivo in [ARQUIVO, "melhor_modelo.keras"]:
    if Path(arquivo).exists():
        print("Baixando:", arquivo)
        files.download(arquivo)
    else:
        print(f"[!] Não existe, pulando: {arquivo}")

---
# 9. Conclusões

## O que foi construído

Uma CNN treinada do zero que classifica imagens de 32×32 em 100 categorias,
avaliada não só por acurácia mas pela **estrutura semântica dos seus erros**.

## Os três resultados que importam

**1. Top-5 conta uma história diferente de top-1.** A diferença entre as duas
métricas é grande, e mostra que em boa parte dos casos o modelo tem a resposta
certa entre seus melhores palpites. Para uma aplicação real — sugerir etiquetas
a um usuário, por exemplo — isso pode ser suficiente.

**2. Os erros não são aleatórios.** A proporção de erros que caem na
superclasse correta fica muito acima dos ~4% que o acaso explicaria. O modelo
aprendeu que golfinho se parece mais com baleia do que com guarda-roupa, sem
que ninguém tenha lhe ensinado a hierarquia — ela nunca entrou no treino.

**3. Superclasses visuais vencem superclasses funcionais.** Famílias unidas por
aparência (`trees`, `flowers`) têm desempenho muito superior às unidas por
função (`household_electrical_devices`). Faz sentido: uma CNN enxerga forma,
cor e textura — não utilidade. É uma limitação estrutural da abordagem, não um
defeito deste modelo.

## Limitações honestas

- **32×32 é pouquíssima informação.** Muitas imagens são ambíguas até para uma
  pessoa. Parte do erro é irredutível sem imagens melhores.
- **500 exemplos por classe é pouco** para 100 categorias. É o principal
  limitador da acurácia aqui.
- **A superclasse não entra no treino.** Ela é usada apenas para analisar os
  erros. Treinar com as duas saídas ao mesmo tempo é uma extensão natural.
- **Acurácia não é tudo.** Um modelo com 60% que erra "perto" pode ser mais útil
  que um com 65% que erra "longe", dependendo da aplicação.

## Extensões possíveis

- **Transfer learning:** redimensionar para 128×128 e usar EfficientNetB0
  pré-treinado no ImageNet passa de 80% de acurácia, ao custo de mais tempo
- **Saída dupla:** treinar simultaneamente para classe fina e superclasse, com
  duas cabeças e perda combinada — a hierarquia deixaria de ser só análise e
  passaria a guiar o aprendizado
- **Conexões residuais:** um ResNet pequeno permite ir mais fundo sem que o
  gradiente se perca
- **Agendamento de taxa de aprendizado:** um decaimento por cosseno costuma
  render 2 a 3 pontos a mais que o `ReduceLROnPlateau`